# AeroPredict: project walkthrough

This notebook uses the unchanged 6,000 synthetic records and the saved offline RF pipelines. The 3D website uses separate simulation rules. No live API requests are made.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import joblib
from IPython.display import display, Image

ROOT = Path.cwd()
if not (ROOT / 'dataset').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'dataset/metadata.json').exists(), 'Open from ML_Project or its notebooks folder.'
meta = json.loads((ROOT / 'dataset/metadata.json').read_text())
frames = {s: pd.read_csv(ROOT / 'dataset' / f'{s}.csv') for s in ['train', 'validation', 'test']}
metrics = json.loads((ROOT / 'results/metrics.json').read_text())


## Dataset and exact features

The labels are `delayed_15_plus` and `departure_delay_minutes`. Neither target is an input. Origin is always HYD but remains in the original feature list.


In [ ]:
display(pd.DataFrame({'split': list(frames), 'rows': [len(x) for x in frames.values()]}))
display(pd.DataFrame({'feature': meta['features']}))
display(frames['train'].head())


## Training and preprocessing

The saved pipeline fits one-hot encoding on training data only for origin, destination and weather, and passes through the seven numeric features. Validation selects parameters. Testing uses 900 held-out records. To repeat the study, run `python src/ml/train_evaluate.py` from the project root. This notebook does not retrain automatically.


In [ ]:
classifier = joblib.load(ROOT / 'models/random_forest_classification.joblib')
regressor = joblib.load(ROOT / 'models/random_forest_regression.joblib')
print('Encoded feature count:', len(regressor.named_steps['preprocess'].get_feature_names_out()))
print('Trees:', regressor.named_steps['model'].n_estimators)
print('Maximum depth:', regressor.named_steps['model'].max_depth)


## Results and worked examples

The two example scenarios have no observed outcome. RF probabilities are raw model outputs, not calibrated confidence. Website rule estimates and RF estimates come from separate implementations.


In [ ]:
display(pd.read_csv(ROOT / 'results/comparison.csv'))
examples = pd.read_csv(ROOT / 'results/worked_examples.csv')
x = examples[meta['features']]
answer = x.copy()
answer['rf_predicted_class'] = classifier.predict(x)
answer['rf_delay_probability'] = classifier.predict_proba(x)[:, 1]
answer['rf_predicted_minutes'] = regressor.predict(x)
display(answer)


## Graphs

Figures are generated from the submitted CSVs and measured metrics. Run `python src/ml/make_graphs.py` to regenerate them.


In [ ]:
for name in ['delay_distribution.png', 'delay_by_weather.png', 'confusion_matrix.png', 'model_comparison.png']:
    display(Image(filename=str(ROOT / 'graphs' / name)))


## Website and code map

- `src/ml/train_evaluate.py`: offline training and evaluation.
- `src/ml/predict.py`: saved-model prediction.
- `src/website/server.mjs` and `server/`: local server, private settings, cache and provider handling.
- `src/website/public/simulation/engine.mjs`: current simulation rules.
- `globe.mjs`: camera and route presentation.
- `airport-scene.mjs` and the Blender sources under `src/website/airport/`: airport journey.

Top, Side, Along route, World and Follow change the camera, not the predicted delay. Provider schedules and valid position snapshots have a different role from synthetic training data.
